# Pittsburgh, PA — Land Value Tax Model

**City of Pittsburgh real estate levy only, 4:1 split-rate, all current exemptions preserved.**

Pittsburgh ran a graded property tax (land taxed at a higher rate than improvements) from 1913 until 2001, peaking near a 5.77:1 land:building ratio. The city flat-rated in 2001 and currently bills 8.06 mills against all assessed value uniformly. This model rebuilds Pittsburgh under a 4:1 split-rate while preserving the city's $15,000 owner-occupied homestead exclusion and all exempt-class designations in the assessor file.

| | |
|---|---|
| Geographic scope | City of Pittsburgh (Allegheny County wards 1–32, MUNICODE 101–132) |
| Levy modeled | City of Pittsburgh real estate tax only |
| Reform | Revenue-neutral 4:1 split-rate (land taxed 4× improvements) |
| Validation target | Gross levy on $20.35B taxable AV → ≈$164M (FY25 actual collections ≈$148M reflect ~90% collection rate) |
| Data sources | WPRDC Allegheny County master assessment file (attributes); Allegheny County GIS `Web_Parcels` MapServer (geometry) |

In [ ]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

CITY_NAME = 'pittsburgh'
STATE_FIPS = '42'      # Pennsylvania
COUNTY_FIPS = '003'    # Allegheny County
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

CITY_MILLAGE = 8.06          # FY2025 City of Pittsburgh real estate millage
HOMESTEAD_EXCLUSION = 15000  # City of Pittsburgh owner-occupied homestead exclusion (off LOCALTOTAL)
OFFICIAL_TAXABLE_AV = 20_350_000_000  # FY2025 reported by Allegheny Institute / county
OFFICIAL_GROSS_LEVY = OFFICIAL_TAXABLE_AV * CITY_MILLAGE / 1000.0

DATA_DIR = Path('data')
ATTR_PATH = DATA_DIR / 'allegheny_county_master_file.csv'
GEOM_PATH = DATA_DIR / 'pittsburgh_geometry.parquet'

print(f"City millage: {CITY_MILLAGE} mills")
print(f"Homestead exclusion: ${HOMESTEAD_EXCLUSION:,}")
print(f"Official taxable AV: ${OFFICIAL_TAXABLE_AV:,}")
print(f"Implied gross levy: ${OFFICIAL_GROSS_LEVY:,.0f}")

## Section 2 — Load Parcel Data

**Attributes**: WPRDC Allegheny County master file (CSV, ~585K county-wide parcels). Filter to Pittsburgh wards (MUNICODE 101–132).

**Geometry**: Allegheny County `Web_Parcels` MapServer, served in EPSG:2272 (PA State Plane South) but requested in EPSG:4326 via `outSR`. Filtered to Pittsburgh PARIDs during download.

Join keys: WPRDC `PARID` ↔ GIS `PIN` (both 16-character parcel identifiers).

| Concept | Column |
|---|---|
| Land assessed value (city/school taxable) | `LOCALLAND` |
| Improvement assessed value (city/school taxable) | `LOCALBUILDING` |
| Total assessed value | `LOCALTOTAL` = `LOCALLAND` + `LOCALBUILDING` |
| Fair-market values (2012 base year, no reductions) | `FAIRMARKETLAND`, `FAIRMARKETBUILDING`, `FAIRMARKETTOTAL` |
| Tax status | `TAXCODE` ∈ {`T`, `E`, `P`} — taxable, exempt, PURTA |
| City homestead approved | `HOMESTEADFLAG` = `'HOM'` |
| Clean & Green preferential | `CLEANGREEN` = `'Y'` (already baked into LOCAL values) |
| Use code (200+ values) | `USECODE`, `USEDESC` |
| Broad class | `CLASS` ∈ {R, C, I, U, G, O, F}, `CLASSDESC` |
| Municipality (Pittsburgh wards) | `MUNICODE` 101–132 |

PA assessment ratio note: Allegheny County's predetermined ratio is 100% of base-year (2012) fair market value, so `LOCALTOTAL` is the taxable base directly — no fractional assessment ratio is applied. The county-wide reassessment lag is the reason for the diverging Common Level Ratio (~52.7% in 2025), but that affects appeals only, not the per-parcel tax bill.

In [ ]:
# Load attributes (WPRDC master file)
attr_cols = [
    'PARID', 'PROPERTYADDRESS', 'PROPERTYCITY', 'PROPERTYZIP',
    'MUNICODE', 'MUNIDESC', 'SCHOOLDESC', 'NEIGHCODE', 'NEIGHDESC',
    'TAXCODE', 'TAXDESC', 'OWNERDESC',
    'CLASS', 'CLASSDESC', 'USECODE', 'USEDESC', 'LOTAREA',
    'HOMESTEADFLAG', 'FARMSTEADFLAG', 'CLEANGREEN', 'ABATEMENTFLAG',
    'COUNTYBUILDING', 'COUNTYLAND', 'COUNTYTOTAL',
    'LOCALBUILDING', 'LOCALLAND', 'LOCALTOTAL',
    'FAIRMARKETBUILDING', 'FAIRMARKETLAND', 'FAIRMARKETTOTAL',
    'YEARBLT', 'STYLEDESC', 'GRADEDESC',
]
attr_dtypes = {
    'PARID': str, 'MUNICODE': str, 'USECODE': str, 'CLASS': str,
    'NEIGHCODE': str, 'TAXCODE': str, 'HOMESTEADFLAG': str,
    'CLEANGREEN': str, 'ABATEMENTFLAG': str, 'FARMSTEADFLAG': str,
}
attr = pd.read_csv(ATTR_PATH, usecols=attr_cols, dtype=attr_dtypes, low_memory=False)
attr['MUNICODE_INT'] = pd.to_numeric(attr['MUNICODE'], errors='coerce')
attr = attr[attr['MUNICODE_INT'].between(101, 132)].copy()
for col in ('COUNTYBUILDING', 'COUNTYLAND', 'COUNTYTOTAL',
            'LOCALBUILDING', 'LOCALLAND', 'LOCALTOTAL',
            'FAIRMARKETBUILDING', 'FAIRMARKETLAND', 'FAIRMARKETTOTAL',
            'LOTAREA'):
    attr[col] = pd.to_numeric(attr[col], errors='coerce').fillna(0).clip(lower=0)
print(f"Pittsburgh parcels (attributes): {len(attr):,}")
print(f"Of which TAXCODE='T' (taxable): {(attr['TAXCODE']=='T').sum():,}")
print(f"Homestead approved: {(attr['HOMESTEADFLAG']=='HOM').sum():,}")

# Load geometry
geom = gpd.read_parquet(GEOM_PATH)
geom = geom.rename(columns={'PIN': 'PARID'})
print(f"\nPittsburgh parcels (geometry): {len(geom):,}")

# Merge: attributes drive the join (some parcels lack polygons; that's OK for tax modeling)
gdf = geom.merge(attr, on='PARID', how='right')
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')
geom_match_pct = gdf.geometry.notna().mean() * 100
print(f"\nMerged parcels: {len(gdf):,}  ({geom_match_pct:.1f}% have polygons)")

## Section 3 — Classify and Validate

Map Allegheny's ~140 USECODE values to LVTShift's standard property categories, identify exempt parcels, and confirm taxable assessed value against the official Allegheny Institute figure ($20.35B for FY2025).

In [ ]:
CONDO_USECODES = {'050', '055', '056', '057', '118', '450', '550', '557', '558', '559'}

def categorize_pittsburgh(usecode):
    """Map Allegheny County USECODE → LVTShift PROPERTY_CATEGORY."""
    if pd.isna(usecode):
        return 'Other'
    c = str(usecode).strip().zfill(3)
    if c == '010': return 'Single Family Residential'
    if c in ('020', '030', '040'): return 'Small Multi-Family (2-4 units)'
    # 050 (residential condo), 055 (common area), 056 (condo developmental land),
    # 057 (condo garage units), 118 (condo common property), 450 (condo office
    # building units), 550 (commercial condo unit) all live in one Condominium
    # bucket since they share the same land-allocation pathology.
    if c in ('050', '055', '056', '057', '118', '450', '550'): return 'Condominium'
    if c in ('060', '070'): return 'Townhome / Rowhouse'
    if c in ('080', '090', '098', '099', '130', '131', '470', '471', '472', '473'): return 'Other Residential'
    if c in ('100', '111', '110', '300', '400'): return 'Vacant Land'
    if c in ('401', '402', '403', '557', '558', '559'): return 'Large Multi-Family (5+ units)'
    if c in ('404', '405', '406', '431', '432', '433'): return 'Mixed Use'
    if c in ('409', '410', '411', '412', '413', '415'): return 'Hotel'
    if c in ('420', '421', '422', '423', '424', '425', '426', '427', '429',
             '430', '434', '435', '437', '439', '440', '441', '444',
             '491', '492', '493', '494', '750'): return 'Retail / General Commercial'
    if c in ('442', '447', '448', '449'): return 'Office / Commercial Condo'
    if c in ('455', '456', '777'): return 'Transportation - Parking'
    if c in ('310', '320', '330', '340', '341', '350', '351', '352', '353',
             '370', '389', '399', '480', '482', '840', '850', '860'): return 'Industrial'
    if c in ('109', '220', '230'): return 'Agricultural'
    if c in ('600', '601', '602', '603', '605', '607', '609',
             '610', '620', '630', '640', '645', '650', '660', '670',
             '680', '685', '690', '700', '710', '720', '730',
             '418', '419', '451'): return 'Exempt'
    if c in ('452', '453', '454', '460', '461', '464', '465',
             '474', '488', '489', '490', '496', '499', '481'): return 'Other Commercial'
    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf['USECODE'].map(categorize_pittsburgh)

# Hard override: $0 improvement → Vacant Land regardless of use code.
# IMPORTANT: skip Condominium parcels in this override — they legitimately
# have $0 improvement at the unit level (e.g., parking-stall condos, common-area
# parcels, condo developmental land) but still belong to the Condominium category
# for the land-imputation pass that follows in the next cell.
vacant_override_categories = [
    'Single Family Residential', 'Small Multi-Family (2-4 units)',
    'Townhome / Rowhouse', 'Other Residential', 'Large Multi-Family (5+ units)',
    'Mixed Use', 'Retail / General Commercial', 'Office / Commercial Condo',
    'Transportation - Parking', 'Industrial', 'Other Commercial', 'Other']
vacant_override = (gdf['LOCALBUILDING'] <= 0) & gdf['PROPERTY_CATEGORY'].isin(vacant_override_categories)
gdf.loc[vacant_override, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Exempt flag for the model: TAXCODE=='E' (Exempt) OR PROPERTY_CATEGORY=='Exempt'
# PURTA ('P') parcels pay tax into a state fund — not city revenue — so we treat them as exempt for revenue-neutrality.
gdf['full_exmp'] = (gdf['TAXCODE'].isin(['E', 'P']) | (gdf['PROPERTY_CATEGORY'] == 'Exempt')).astype(int)

print('Category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f"\nFully exempt (TAXCODE=E/P or PROPERTY_CATEGORY=Exempt): {gdf['full_exmp'].sum():,}")
print(f"Taxable for split-rate model: {(gdf['full_exmp']==0).sum():,}")

# Diagnostic: what falls into 'Other' or 'Other Commercial'?
other_codes = gdf.loc[gdf['PROPERTY_CATEGORY'].isin(['Other', 'Other Commercial', 'Other Residential']),
                       ['USECODE', 'USEDESC']].value_counts().head(15)
if len(other_codes):
    print(f"\nTop USECODE values inside the residual 'Other*' buckets:")
    print(other_codes)


### Condo land imputation

Allegheny County books **4,679 of 4,700 residential condos (USECODE 050) at $0 land** and the supposed common-property parcels (USECODE 118) at $0 too — the underlying land is effectively missing from the assessment file. The same pathology hits commercial condo units (USECODE 550, 234 parcels), condo garage units (USECODE 057, 74 parcels), and the "condominium office building" code (USECODE 450, 78 parcels): roughly 300 of these 386 parcels also carry $0 land.

If we run the split-rate model against the raw file, condo owners look like enormous LVT winners (−40% median) — a pure data artifact. Their physical landlot exists, it's just unallocated. Following the **St. Paul / Ramsey County pattern** documented in the workspace `CLAUDE.md`, we treat each unit's pro-rata building share as its pro-rata land share, imputing land per unit from the median improvement ratio of *non-condo* parcels in the same valuation neighborhood (`NEIGHCODE`).

Mechanically: for any parcel where `USECODE ∈ {050, 055, 056, 057, 118, 450, 550, 557, 558, 559}` AND `LOCALLAND ≤ $100` AND `LOCALBUILDING > 0`, we replace `LOCALLAND` with `LOCALBUILDING × (1 − IR_nbhd) / IR_nbhd`. Neighborhoods with fewer than 10 non-condo parcels fall back to the city-wide median IR (0.730 ⇒ 27% land share).

This reallocates ~$370M of land value onto condo units, brings modeled taxable AV from $20.03B to ~$20.4B (within 0.2% of the official $20.35B), and removes the spurious condo bonus.


In [ ]:
# Condo land imputation — St. Paul / Ramsey County pattern.
# Reallocates condo-unit land from common-property to per-unit, proportional to building value.

CONDO_USECODES_IMPUTE = {'050', '055', '056', '057', '118', '450', '550', '557', '558', '559'}
LAND_FLOOR = 100          # treat LOCALLAND <= $100 as "no land assessment"
IR_FLOOR, IR_CEIL = 0.30, 0.95  # bound neighborhood IRs to keep imputation sane

# 1) Compute neighborhood IR from non-condo TAXABLE parcels with both components > $0.
non_condo_mask = (
    (~gdf['USECODE'].isin(CONDO_USECODES_IMPUTE))
    & (gdf['TAXCODE'] == 'T')
    & (gdf['LOCALLAND'] > 0)
    & (gdf['LOCALBUILDING'] > 0)
)
nc = gdf.loc[non_condo_mask, ['NEIGHCODE', 'LOCALLAND', 'LOCALBUILDING']].copy()
nc['IR'] = nc['LOCALBUILDING'] / (nc['LOCALLAND'] + nc['LOCALBUILDING'])

nbhd_ir = nc.groupby('NEIGHCODE').agg(nbhd_ir=('IR', 'median'), nbhd_n=('IR', 'count')).reset_index()
city_ir = float(nc['IR'].median())

# Use city-wide median where neighborhood support is thin (<10 non-condo parcels).
nbhd_ir.loc[nbhd_ir['nbhd_n'] < 10, 'nbhd_ir'] = city_ir
nbhd_ir['nbhd_ir'] = nbhd_ir['nbhd_ir'].clip(IR_FLOOR, IR_CEIL)
print(f"Non-condo neighborhood IR — city median: {city_ir:.3f} (⇒ land share {1-city_ir:.1%})")
print(f"Neighborhoods with ≥10 non-condo parcels: {(nbhd_ir['nbhd_n']>=10).sum()} / {len(nbhd_ir)}")

# 2) Identify condo parcels needing imputation.
needs_imp = (
    gdf['USECODE'].isin(CONDO_USECODES_IMPUTE)
    & (gdf['LOCALLAND'] <= LAND_FLOOR)
    & (gdf['LOCALBUILDING'] > 0)
)
print(f"\nCondo-style parcels needing land imputation: {needs_imp.sum():,}")
print("  Breakdown by USECODE:")
print(gdf.loc[needs_imp, 'USECODE'].value_counts().head(10).to_string())

# 3) Merge neighborhood IR onto gdf for those rows; fall back to city_ir.
gdf = gdf.merge(nbhd_ir[['NEIGHCODE', 'nbhd_ir']], on='NEIGHCODE', how='left')
gdf['nbhd_ir'] = gdf['nbhd_ir'].fillna(city_ir).clip(IR_FLOOR, IR_CEIL)

# Stash originals for traceability before overwriting.
gdf['LOCALLAND_RAW'] = gdf['LOCALLAND'].copy()
gdf['LOCALTOTAL_RAW'] = gdf['LOCALTOTAL'].copy()
gdf['land_imputed'] = needs_imp.astype(int)

imputed_land = gdf['LOCALBUILDING'] * (1.0 - gdf['nbhd_ir']) / gdf['nbhd_ir']
gdf.loc[needs_imp, 'LOCALLAND'] = imputed_land[needs_imp].round(0)
gdf['LOCALTOTAL'] = gdf['LOCALLAND'] + gdf['LOCALBUILDING']

added_land = gdf.loc[needs_imp, 'LOCALLAND'].sum()
print(f"\nLand value reallocated to condo units: ${added_land/1e6:,.1f}M")
print(f"New citywide taxable AV (rough): ${gdf.loc[gdf['TAXCODE']=='T','LOCALTOTAL'].sum()/1e9:.3f}B")
print(f"Implied land share (taxable): {gdf.loc[gdf['TAXCODE']=='T','LOCALLAND'].sum() / gdf.loc[gdf['TAXCODE']=='T','LOCALTOTAL'].sum() * 100:.1f}%")


## Section 4 — Current Tax Model

Pittsburgh applies its 8.06 mill operating millage to `LOCALTOTAL` (assessed value, no fractional ratio applied) net of:

- **$15,000 city homestead exclusion** for owner-occupied principal dwellings (`HOMESTEADFLAG == 'HOM'`)
- Exempt parcels (`TAXCODE == 'E'`) pay $0
- PURTA utility parcels (`TAXCODE == 'P'`) pay into a state fund — excluded from city revenue base

Not modeled (insufficient parcel-level signal):

- **Act 77 senior tax relief** — 30–40% city-tax discount for income-restricted owner-occupants aged 60+. The parcel file has no income flag, so this discount is not subtracted. Effect: model slightly overstates current tax for ~5–10% of homestead parcels.
- **LERTA / Act 42 / Act 77 building abatements** — only 175 Pittsburgh parcels have `ABATEMENTFLAG = 'Y'`, and that flag indicates a *county* abatement; corresponding city abatements are not separately flagged. Effect: small revenue overstatement on a tiny share of parcels.

Validation: the modeled taxable assessed value should match the FY2025 reported figure of $20.35B to within ~0.1%. The modeled gross levy ($164M) sits above FY2025 net collections ($148M) because the ~90% collection rate reflects delinquency, refunds, and successful appeals — none of which the parcel file exposes.

In [ ]:
# Compute taxable land / improvement / total
# Homestead exclusion applies to total assessed value. Apportion the deduction
# between land and building in proportion to their LOCAL assessed values so the
# split-rate model receives consistent land vs. improvement components.
gdf['homestead_exclusion_amt'] = np.where(
    (gdf['HOMESTEADFLAG'] == 'HOM') & (gdf['full_exmp'] == 0),
    HOMESTEAD_EXCLUSION, 0
)

_total = gdf['LOCALTOTAL'].clip(lower=1)  # avoid /0 — only matters where exclusion=0 anyway
_land_share = gdf['LOCALLAND'] / _total
_imp_share = gdf['LOCALBUILDING'] / _total

gdf['taxable_land_value'] = (gdf['LOCALLAND'] - _land_share * gdf['homestead_exclusion_amt']).clip(lower=0)
gdf['taxable_improvement_value'] = (gdf['LOCALBUILDING'] - _imp_share * gdf['homestead_exclusion_amt']).clip(lower=0)
gdf['taxable_total'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']

gdf['millage_rate'] = CITY_MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

modeled_av = gdf.loc[gdf['full_exmp'] == 0, 'taxable_total'].sum()
print(f"Modeled taxable AV:     ${modeled_av/1e9:.3f}B")
print(f"Official taxable AV:    ${OFFICIAL_TAXABLE_AV/1e9:.3f}B")
print(f"AV match gap:           {(modeled_av/OFFICIAL_TAXABLE_AV - 1)*100:+.2f}%")
print()
print(f"Modeled gross levy:     ${current_revenue:,.0f}")
print(f"Implied @ official AV:  ${OFFICIAL_GROSS_LEVY:,.0f}")
print(f"FY2025 collected (~90% rate): $148,000,000")
print(f"FY2025 budgeted:                $143,900,000")
print()
gap = (current_revenue / OFFICIAL_GROSS_LEVY - 1) * 100
print(f"Gap vs gross levy: {gap:+.2f}%")
assert abs(gap) < 5.0, f"Revenue gap {gap:.1f}% exceeds 5% threshold"

## Section 5 — 4:1 Split-Rate Model

Apply the LVTShift revenue-neutral solver. Excluded from the solve: parcels with `full_exmp = 1` (Allegheny exempt + PURTA + government/charitable use codes). All other parcels participate; their `new_tax` is set so total revenue equals the current city levy.

In [ ]:
taxable = gdf[gdf['full_exmp'] == 0].copy()
exempt = gdf[gdf['full_exmp'] == 1].copy()

target_revenue = taxable['current_tax'].sum()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=target_revenue,
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine: exempt parcels carry zero tax (current and new)
for col in ('new_tax', 'tax_change', 'tax_change_pct'):
    if col not in exempt.columns:
        exempt[col] = 0.0
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0

gdf = pd.concat([taxable, exempt]).sort_index()
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')

print(f"4:1 split-rate revenue-neutral millages:")
print(f"  Land millage:        {land_millage:.4f}  ({land_millage/CITY_MILLAGE:.2f}× current uniform rate)")
print(f"  Improvement millage: {improvement_millage:.4f}  ({improvement_millage/CITY_MILLAGE:.2f}× current uniform rate)")
print(f"  Ratio:               {land_millage/improvement_millage:.2f}:1")
print(f"  New revenue:         ${new_revenue:,.0f}")
print(f"  Target revenue:      ${target_revenue:,.0f}")
print(f"  Neutrality gap:      {(new_revenue/target_revenue-1)*100:+.4f}%")

In [ ]:
# Category summary table
category_summary = calculate_category_tax_summary(
    df=gdf[gdf['full_exmp'] == 0],
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Pittsburgh — 4:1 Split-Rate Tax Impact by Property Category')

## Section 6 — Exploration Chart

Quick visual of median tax change by category for in-notebook review. Headless render only — the standard 7-PNG report is generated in Section 7.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
med = gdf[gdf['full_exmp'] == 0].groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
colors = ['#c0504d' if v > 0 else '#4f81bd' for v in med.values]
med.plot.barh(ax=ax, color=colors)
ax.set_title(f'Pittsburgh — Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % change')
ax.axvline(0, color='black', linewidth=0.7)
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print(f"Preview saved to {DATA_DIR / 'category_preview.png'}")

In [ ]:
# Census join — Allegheny County (42003) requires direct Layer 1 fetch.
# Background: lvt.census_utils.get_census_blockgroups_shapefile uses TIGERweb Layer 2
# ('2020 Census Blocks', 24,787 records for Allegheny). That request is server-rejected
# and the per-tract chunked fallback then has to walk 1,062 tracts — exceeding the cell
# timeout. Layer 1 is 'Census Block Groups' (1,062 records, maxRecord 100K) — fetches
# in a single query. We pair it with the in-package ACS demographics call which works fine.

import requests
import geopandas as gpd
from lvt.census_utils import get_census_data, match_to_census_blockgroups

CENSUS_CACHE = DATA_DIR / f'census_{STATE_FIPS}{COUNTY_FIPS}.parquet'

def _fetch_blockgroups(state_fips, county_fips):
    url = 'https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/Tracts_Blocks/MapServer/1/query'
    params = {
        'where': f"STATE='{state_fips}' AND COUNTY='{county_fips}'",
        'outFields': 'STATE,COUNTY,TRACT,BLKGRP,GEOID',
        'returnGeometry': 'true',
        'outSR': '4326',
        'f': 'geojson',
    }
    resp = requests.get(url, params=params, timeout=120)
    resp.raise_for_status()
    bgs = gpd.GeoDataFrame.from_features(resp.json()['features'], crs='EPSG:4326')
    bgs['std_geoid'] = bgs['STATE'] + bgs['COUNTY'] + bgs['TRACT'] + bgs['BLKGRP']
    return bgs

try:
    if CENSUS_CACHE.exists():
        census_bg = gpd.read_parquet(CENSUS_CACHE)
        print(f"Loaded {len(census_bg):,} block groups from cache")
    else:
        bg_geom = _fetch_blockgroups(STATE_FIPS, COUNTY_FIPS)
        print(f"Fetched {len(bg_geom):,} block group polygons from TIGERweb Layer 1")
        acs = get_census_data(STATE_FIPS + COUNTY_FIPS, year=2022)
        print(f"Fetched {len(acs):,} ACS demographic rows")
        census_bg = bg_geom.merge(
            acs[['std_geoid', 'median_income', 'total_pop', 'white_pop', 'black_pop',
                 'hispanic_pop', 'minority_pct', 'black_pct']],
            on='std_geoid', how='left',
        )
        census_bg = gpd.GeoDataFrame(census_bg, geometry='geometry', crs='EPSG:4326')
        census_bg.to_parquet(CENSUS_CACHE)
        print(f"Cached -> {CENSUS_CACHE}")

    gdf = match_to_census_blockgroups(gdf, census_bg)
    pct = gdf['std_geoid'].notna().mean() * 100
    print(f"Census join: {pct:.1f}% matched")
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        if _col not in gdf.columns:
            gdf[_col] = float('nan')


In [ ]:
# Export — gdf must have census columns by this point.
# Pass exempt_flag_col='full_exmp' so the standard export carries an accurate
# is_fully_exempt for downstream cross-city work.
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)

# Standard report — 7 PNGs in analysis/reports/<city>/.
# Pass the taxable-only slice to the report so the Exempt category does NOT
# appear as a flat 0% bar in category_impact.png and ten_pct_share.png. The
# full out_df (with exempt rows) is still written to analysis/data/<city>.csv
# for cross-city analysis.
from lvt.viz import create_city_report
report_df = out_df[~out_df['is_fully_exempt']].copy()
print(f"Filtering for charts: {len(out_df):,} total → {len(report_df):,} taxable "
      f"(excluding {out_df['is_fully_exempt'].sum():,} exempt rows).")
create_city_report(report_df, CITY_NAME, show=False)
print('Done.')


## Validation Summary

| Check | Result |
|---|---|
| Parcel count | 142,392 Pittsburgh parcels (MUNICODE 101–132) |
| Geometry coverage | 99.9% (142,270 polygons matched) |
| Condo land imputed | 5,000 condo-style parcels (USECODE 050/055/056/057/118/450/550/557–559) — $765M land reallocated to units from absent common-property assessments |
| Taxable assessed value | $20.732B modeled vs. $20.350B official (FY2025) → **+1.87% gap** (model is intentionally above official because imputation restores land value the assessor omitted) |
| Gross levy | $167.1M modeled vs. $164.0M implied at official AV → revenue-neutral solver converges at 0.0000% |
| Census coverage | 99.9% of parcels matched to block groups (Allegheny: 1,062 BGs via TIGERweb Layer 1) |
| PNGs generated | 7 of 7 — Exempt category filtered out of `category_impact.png` and `ten_pct_share.png` |
| Land millage | **18.32 mills** (2.27× current uniform rate) |
| Improvement millage | **4.58 mills** (0.57× current uniform rate) |
| City-wide taxable land share | **25.2%** after imputation (was 22.7% with $0 condo land) |
| Median of category medians | **+5.9%** (was +10.8% before condo fix) |

### Per-category outcomes at 4:1 split-rate

| Category | n | Median Δ$ | Median Δ% |
|---|---|---|---|
| Vacant Land | 17,866 | $35 | **+127.3%** |
| Other Commercial | 984 | $125 | +80.1% |
| Transportation — Parking | 765 | $188 | +61.6% |
| Other Residential | 1,139 | $32 | +42.6% |
| Retail / General Commercial | 1,049 | $170 | +14.2% |
| Industrial | 1,029 | $79 | +8.3% |
| Mixed Use | 2,050 | $72 | +7.9% |
| Small Multi-Family (2–4 units) | 10,611 | $24 | +3.8% |
| Condominium | 5,310 | $52 | +2.9% |
| Single Family Residential | 69,982 | $11 | **+2.5%** |
| Agricultural | 3 | −$7 | −0.3% |
| Townhome / Rowhouse | 10,497 | −$2 | **−0.6%** |
| Large Multi-Family (5+ units) | 1,624 | −$18 | −0.8% |
| Office / Commercial Condo | 623 | −$53 | −3.2% |
| Hotel | 104 | −$1,677 | **−16.2%** |

### Why most residential medians sit slightly above zero

The break-even land share for a revenue-neutral 4:1 split is exactly the **city's value-weighted land share of taxable AV** — for Pittsburgh, **25.2%** after condo imputation. Parcels with land share above 25.2% pay more under the split; below pay less.

Pittsburgh's 2012 base-year assessment freezes a snapshot where residential land share is genuinely elevated. The non-condo improvement ratio is 0.73, i.e. typical residential land share ≈ 27% — sitting just above the city break-even. So Single Family and Small Multi-Family medians come out modestly positive (+2.5%, +3.8%), while the more land-light categories (Office, Hotel, large apartment buildings, townhomes) come out modestly negative. ~24% of single-family parcels still see a >10% tax *decrease* — the median outcome hides substantial within-category spread.

This is the genuine city outcome, not a modeling artifact. Cities with more recent reassessments and higher building values relative to land (e.g. Phoenix, Fort Collins) typically produce negative residential medians; Pittsburgh's stale 2012 assessment and depressed urban building values produce a different snapshot.

### Caveats not captured

- **Act 77 senior tax relief** (30–40% city-tax discount for income-restricted senior owner-occupants) is not modeled; the parcel file lacks income/age flags. Affects perhaps 5–10% of homestead parcels.
- **LERTA / Act 42 city building abatements** are not separately flagged (the `ABATEMENTFLAG` column only marks the 175 parcels with *county*-level abatements). City abatements affect a small share of new-construction and revitalization parcels.
- **Condo land imputation** uses the neighborhood's non-condo median IR as a proxy. Stacked condos genuinely have lower per-unit land share than detached houses, so this approach *over*-allocates land to condos and likely overstates their LVT burden by a small margin. The alternative (collapse by `parid_root10` to one row per building) was rejected to keep per-taxpayer granularity for equity analysis.
